# 6TH : Linear classification

In [41]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold

In [ ]:
def load_cifar10_data(file_path):
    with open(file_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data']
    labels = batch[b'labels']
    return images, np.array(labels)

data_dir = 'cifar-10-batches-py'

all_train_images = []
all_train_labels = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    images, labels = load_cifar10_data(batch_path)
    all_train_images.append(images)
    all_train_labels.append(labels)

all_train_images = np.concatenate(all_train_images, axis=0)
all_train_labels = np.concatenate(all_train_labels, axis=0)

test_path = os.path.join(data_dir, 'test_batch')
all_test_images, all_test_labels = load_cifar10_data(test_path)

print(f"Train dataset: {all_train_images.shape}")
print(f"Test dataset: {all_test_images.shape}")

Train dataset: (50000, 3072)
Test dataset: (10000, 3072)


In [ ]:
class LinearClassifier:
    def __init__(self, input_dim, num_classes):
        self.W = np.random.randn(num_classes, input_dim)
        self.b = np.zeros(num_classes)
        
    def dot(self, X):
        return (np.dot(self.W, X.T)).T + self.b
        
    def softmax(self, scores):
        shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(shifted_scores)
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return probs
        
    def predict(self, X):
        scores = self.dot(X)
        probs = self.softmax(scores)
        return np.argmax(probs, axis=1)

In [48]:
input_dim = all_train_images.shape[1]
num_classes = 10

lc = LinearClassifier(input_dim, num_classes)

preds = lc.predict(all_test_images)
accuracy = np.mean(preds == all_test_labels)
print(f"Prediction Accuracy (without normalization): {accuracy * 100:.2f}%")

Prediction Accuracy (without normalization): 8.36%


In [45]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

print("Starting 5-Fold Cross-Validation on Train Dataset...")

for fold, (train_idx, val_idx) in enumerate(kf.split(all_train_images)):
    X_train_fold, X_val_fold = all_train_images[train_idx], all_train_images[val_idx]
    y_train_fold, y_val_fold = all_train_labels[train_idx], all_train_labels[val_idx]

    lc_fold = LinearClassifier(input_dim, num_classes)
    
    fold_preds = lc_fold.predict(X_val_fold)
    acc = np.mean(fold_preds == y_val_fold)
    fold_accuracies.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc * 100:.2f}%")

print(f"\nOverall Mean Accuracy (5-Fold CV): {np.mean(fold_accuracies) * 100:.2f}%")

Starting 5-Fold Cross-Validation on Train Dataset...
Fold 1 Accuracy: 10.43%
Fold 2 Accuracy: 11.08%
Fold 3 Accuracy: 10.65%
Fold 4 Accuracy: 10.20%
Fold 5 Accuracy: 9.58%

Overall Mean Accuracy (5-Fold CV): 10.39%


In [47]:
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
cm = confusion_matrix(all_test_labels, preds)

cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

cm_df

,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
airplane,1,1,0,230,230,523,3,2,10,0
automobile,0,1,1,306,322,324,12,17,17,0
bird,1,2,1,462,91,438,2,2,1,0
cat,0,1,0,473,89,420,6,4,7,0
deer,0,0,0,440,58,494,2,1,5,0
dog,0,4,0,465,75,435,4,10,7,0
frog,0,0,2,540,62,379,8,4,5,0
horse,0,0,1,400,137,455,4,0,3,0
ship,0,0,0,316,372,289,2,3,18,0
truck,0,2,1,332,334,298,2,6,25,0
